# Online Retail II — Exploratory Data Analysis

Dataset: `online_retail_II.xlsx` — two sheets (`Year 2009-2010`, `Year 2010-2011`),
one row per order line. Columns: `Invoice`, `StockCode`, `Description`, `Quantity`,
`InvoiceDate`, `Price`, `Customer ID`, `Country`.

This notebook loads the data, checks data quality, flags cancellations/returns,
builds a cleaned sales table, and answers the usual first questions:
top customers, top products, products with the most cancellations, revenue by
country, sales trend over time, and basic RFM (recency/frequency/monetary) stats.

In [5]:
%pip install pandas
%pip install matplotlib
%pip install openpyxl

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 250.9/250.9 kB 6.0 MB/s eta 0:00:007.0 MB/s eta 0:00:01
Note: you may need to restart the kernel to use updated packages.


In [6]:
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda x: f'{x:,.2f}')

FILE = 'online_retail_II.xlsx'  

sheets = pd.read_excel(FILE, sheet_name=None)
df = pd.concat(
    [d.assign(SheetYear=name) for name, d in sheets.items()],
    ignore_index=True
)
print('Combined shape:', df.shape)
df.head()

Combined shape: (1067371, 9)


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,SheetYear
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,"13,085.00",United Kingdom,Year 2009-2010
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,"13,085.00",United Kingdom,Year 2009-2010
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,"13,085.00",United Kingdom,Year 2009-2010
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,"13,085.00",United Kingdom,Year 2009-2010
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,"13,085.00",United Kingdom,Year 2009-2010


## 1. Basic overview

In [7]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1067371 entries, 0 to 1067370
Data columns (total 9 columns):
 #   Column       Non-Null Count    Dtype         
---  ------       --------------    -----         
 0   Invoice      1067371 non-null  object        
 1   StockCode    1067371 non-null  object        
 2   Description  1062989 non-null  object        
 3   Quantity     1067371 non-null  int64         
 4   InvoiceDate  1067371 non-null  datetime64[us]
 5   Price        1067371 non-null  float64       
 6   Customer ID  824364 non-null   float64       
 7   Country      1067371 non-null  str           
 8   SheetYear    1067371 non-null  str           
dtypes: datetime64[us](1), float64(2), int64(1), object(3), str(2)
memory usage: 73.3+ MB


In [10]:
df.describe(include='all').T

,count,unique,top,freq,mean,min,25%,50%,75%,max,std
Invoice,"1,067,371.00","53,628.00","537,434.00","1,350.00",NaN,NaN,NaN,NaN,NaN,NaN,NaN
StockCode,1067371,5305,85123A,5829,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Description,1062989,5698,WHITE HANGING HEART T-LIGHT HOLDER,5918,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Quantity,"1,067,371.00",NaN,NaN,NaN,9.94,"-80,995.00",1.00,3.00,10.00,"80,995.00",172.71
InvoiceDate,1067371,NaN,NaN,NaN,2011-01-02 21:13:55.394029,2009-12-01 07:45:00,2010-07-09 09:46:00,2010-12-07 15:28:00,2011-07-22 10:23:00,2011-12-09 12:50:00,NaN
Price,"1,067,371.00",NaN,NaN,NaN,4.65,"-53,594.36",1.25,2.10,4.15,"38,970.00",123.55
Customer ID,"824,364.00",NaN,NaN,NaN,"15,324.64","12,346.00","13,975.00","15,255.00","16,797.00","18,287.00","1,697.46"
Country,1067371,43,United Kingdom,981330,NaN,NaN,NaN,NaN,NaN,NaN,NaN
SheetYear,1067371,2,Year 2010-2011,541910,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 2. Missing values

`Customer ID` is missing for a large share of rows — these are typically guest /
unidentified purchases and must be excluded from any customer-level analysis
(top customers, RFM, etc.), but they can stay in for product/country/revenue totals.

In [11]:
missing = df.isna().sum().to_frame('missing_count')
missing['missing_pct'] = (missing['missing_count'] / len(df) * 100).round(2)
missing.sort_values('missing_count', ascending=False)

,missing_count,missing_pct
Customer ID,243007,22.77
Description,4382,0.41
Invoice,0,0.00
Quantity,0,0.00
StockCode,0,0.00
InvoiceDate,0,0.00
Price,0,0.00
Country,0,0.00
SheetYear,0,0.00


## 3. Cancellations and returns

- A **cancellation** is an invoice whose number starts with `C` (the whole order was cancelled).
- A **return** (not necessarily a full cancellation) is any line with a **negative `Quantity`**.

These are worth separating out before computing "real" sales totals.

In [23]:
# we note that invoices values starting with C are cancellations, and negative quantities are returns. We can create two new columns to flag these cases.  
df['IsCancellation'] = df['Invoice'].astype(str).str.startswith('C')
df['IsReturn'] = df['Quantity'] < 0

print('Cancelled invoice lines:', df['IsCancellation'].sum(), f"({df['IsCancellation'].mean()*100:.2f}% of rows)")
print('Negative-quantity lines:', df['IsReturn'].sum(), f"({df['IsReturn'].mean()*100:.2f}% of rows)")
print('Zero or negative price lines:', (df['Price'] <= 0).sum())

Cancelled invoice lines: 19494 (1.83% of rows)
Negative-quantity lines: 22950 (2.15% of rows)
Zero or negative price lines: 6207


### Products with the most cancellations (by quantity cancelled)

In [30]:
cancelled = df[df['IsCancellation']]

top_cancelled_products = (
    cancelled.groupby(['StockCode', 'Description'])['Quantity']
    .sum()                      # Quantity is negative on cancellations
    .abs()
    .sort_values(ascending=False)
    .head(10)
    .rename('UnitsCancelled')
    .reset_index()
)
top_cancelled_products

,StockCode,Description,UnitsCancelled
0,23843,"PAPER CRAFT , LITTLE BIRDIE",80995
1,23166,MEDIUM CERAMIC TOP STORAGE JAR,74494
2,84347,ROTATING SILVER ANGELS T-LIGHT HLDR,18750
3,21088,SET/6 FRUIT SALAD PAPER CUPS,7140
4,21096,SET/6 FRUIT SALAD PAPER PLATES,7008
5,M,Manual,5450
6,16047,POP ART PEN CASE & PENS,5184
7,85110,BLACK SILVER FLOWER T-LIGHT HOLDER,5040
8,37340,MULTICOLOUR SPRING FLOWER MUG,4996
9,16046,TEATIME PEN CASE & PENS,4632


### Products with the most cancellations (by number of cancellation lines)

In [31]:
top_cancelled_by_lines = (
    cancelled.groupby(['StockCode', 'Description'])
    .size()
    .sort_values(ascending=False)
    .head(10)
    .rename('CancellationLines')
    .reset_index()
)
top_cancelled_by_lines

,StockCode,Description,CancellationLines
0,M,Manual,538
1,22423,REGENCY CAKESTAND 3 TIER,347
2,POST,POSTAGE,229
3,22138,BAKING SET 9 PIECE RETROSPOT,211
4,21232,STRAWBERRY CERAMIC TRINKET BOX,184
5,D,Discount,172
6,85123A,WHITE HANGING HEART T-LIGHT HOLDER,135
7,79323W,WHITE CHERRY LIGHTS,121
8,21843,RED RETROSPOT CAKE STAND,111
9,S,SAMPLES,101


## 4. Build a cleaned "real sales" table

For sales/revenue/customer analysis, exclude:
- cancelled invoices (`IsCancellation`)
- negative or zero quantity lines (returns / adjustments)
- zero or negative price lines (data errors, not real sales)

Keep the excluded rows available separately (`df`, `cancelled`) for returns-specific questions.

In [ ]:
df_clean = df[
    (~df['IsCancellation']) &
    (df['Quantity'] > 0) &
    (df['Price'] > 0)
].copy()

df_clean['Revenue'] = df_clean['Quantity'] * df_clean['Price']

print('Rows kept for sales analysis:', len(df_clean), f"({len(df_clean)/len(df)*100:.1f}% of all rows)")
df_clean[['Invoice', 'StockCode', 'Quantity', 'Price', 'Revenue']].head()

## 5. Top 10 customers

Two views: by **total amount spent** and by **number of distinct orders** (invoices).
Both exclude rows with a missing `Customer ID`.

In [ ]:
known_customers = df_clean.dropna(subset=['Customer ID'])

top_customers_spend = (
    known_customers.groupby('Customer ID')['Revenue']
    .sum()
    .sort_values(ascending=False)
    .head(10)
    .reset_index()
)
print('Top 10 customers by total spend:')
top_customers_spend

In [ ]:
top_customers_orders = (
    known_customers.groupby('Customer ID')['Invoice']
    .nunique()
    .sort_values(ascending=False)
    .head(10)
    .rename('NumOrders')
    .reset_index()
)
print('Top 10 customers by number of orders:')
top_customers_orders

## 6. Top products

In [ ]:
top_products_qty = (
    df_clean.groupby(['StockCode', 'Description'])['Quantity']
    .sum()
    .sort_values(ascending=False)
    .head(10)
    .rename('UnitsSold')
    .reset_index()
)
print('Top 10 products by units sold:')
top_products_qty

In [ ]:
top_products_revenue = (
    df_clean.groupby(['StockCode', 'Description'])['Revenue']
    .sum()
    .sort_values(ascending=False)
    .head(10)
    .reset_index()
)
print('Top 10 products by revenue:')
top_products_revenue

## 7. Revenue and orders by country

In [ ]:
country_summary = (
    df_clean.groupby('Country')
    .agg(Revenue=('Revenue', 'sum'), Orders=('Invoice', 'nunique'), Customers=('Customer ID', 'nunique'))
    .sort_values('Revenue', ascending=False)
)
country_summary.head(15)

In [ ]:
country_summary['Revenue'].head(10).plot(kind='barh')
plt.gca().invert_yaxis()
plt.xlabel('Revenue')
plt.title('Top 10 countries by revenue')
plt.tight_layout()
plt.show()

## 8. Sales trend over time (monthly)

In [ ]:
monthly_revenue = (
    df_clean.set_index('InvoiceDate')
    .resample('MS')['Revenue']
    .sum()
)
monthly_revenue.plot(marker='o')
plt.ylabel('Revenue')
plt.title('Monthly revenue')
plt.tight_layout()
plt.show()

## 9. Basket size and order value

How many distinct line items and units are in a typical order, and what's a typical order value?

In [ ]:
order_summary = (
    df_clean.groupby('Invoice')
    .agg(Lines=('StockCode', 'nunique'), Units=('Quantity', 'sum'), OrderValue=('Revenue', 'sum'))
)
order_summary.describe()

## 10. Customer RFM snapshot (Recency, Frequency, Monetary)

A quick look at customer value, often the starting point for segmentation/churn work.

In [ ]:
snapshot_date = df_clean['InvoiceDate'].max() + pd.Timedelta(days=1)

rfm = known_customers.groupby('Customer ID').agg(
    Recency=('InvoiceDate', lambda x: (snapshot_date - x.max()).days),
    Frequency=('Invoice', 'nunique'),
    Monetary=('Revenue', 'sum'),
)
rfm.sort_values('Monetary', ascending=False).head(10)

In [ ]:
rfm.describe()

## 11. Quick data-quality recap before deeper analysis

- `Customer ID` missing on a large share of rows -> customer-level work only covers identified customers.
- `Invoice` values starting with `C` are cancellations; their `Quantity`/`Price` are typically negative and should not be summed into revenue.
- A smaller set of rows has negative/zero `Price` even without being a cancellation (adjustments, manual corrections, possibly data entry errors) — worth a manual look if they matter to the question being asked.
- Two sheets = two calendar years; concatenated above into one `df` with a `SheetYear` column, but `InvoiceDate` is the reliable field for any date-based grouping (not the sheet name).

In [ ]:
df.groupby('SheetYear').agg(
    Rows=('Invoice', 'size'),
    DateMin=('InvoiceDate', 'min'),
    DateMax=('InvoiceDate', 'max'),
)